# Microsoft Agent Framework with OpenBnB MCP Server Integration (C#)

This notebook demonstrates how to use Microsoft Agent Framework in C# with the actual OpenBnB MCP server to search for real Airbnb accommodations using MCP plugins. For LLM Access, it uses Azure AI Foundry.

## Import the Needed Packages

In [ ]:
#r "nuget: Microsoft.AgentFramework.Core, *-*"
#r "nuget: Microsoft.AgentFramework.MCP, *-*"
#r "nuget: Azure.Identity, 1.13.1"

In [ ]:
// Import required namespaces
using System;
using System.Linq;
using System.Threading.Tasks;
using System.Collections.Generic;
using Azure.Identity;
using Microsoft.AgentFramework;
using Microsoft.AgentFramework.MCP;

## Creating the MCP Plugin Connection

We'll connect to the [OpenBnB MCP server](https://github.com/openbnb-org/mcp-server-airbnb) using MCP Stdio Plugin. This server provides Airbnb search functionality through the @openbnb/mcp-server-airbnb package.

## Environment Configuration

Configure Azure AI Foundry settings. Make sure you have the following environment variables set:
- `AZURE_AI_FOUNDRY_MODEL`
- `AZURE_AI_FOUNDRY_PROJECT_ENDPOINT`

In [ ]:
// Load environment variables
var modelDeploymentName = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_MODEL");
var projectEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_PROJECT_ENDPOINT");

Console.WriteLine("🔍 Checking Azure environment variables...");
Console.WriteLine($"✅ AZURE_AI_FOUNDRY_MODEL: {(!string.IsNullOrEmpty(modelDeploymentName) ? "Set" : "NOT set")}");
Console.WriteLine($"✅ AZURE_AI_FOUNDRY_PROJECT_ENDPOINT: {(!string.IsNullOrEmpty(projectEndpoint) ? "Set" : "NOT set")}");

if (string.IsNullOrEmpty(modelDeploymentName) || string.IsNullOrEmpty(projectEndpoint))
{
    throw new InvalidOperationException("Required environment variables are not set.");
}

## Understanding the OpenBnB MCP Integration

This notebook connects to the **real OpenBnB MCP server** that provides actual Airbnb search functionality.

### How it works:

1. **MCP Stdio Plugin**: Uses standard input/output communication with the MCP server
2. **Real NPM Package**: Downloads and runs `@openbnb/mcp-server-airbnb` via npx
3. **Live Data**: Returns actual Airbnb property data from their APIs
4. **Function Discovery**: The agent automatically discovers available functions from the MCP server

### Available Functions:

The OpenBnB MCP server typically provides:
- **search_listings** - Search for Airbnb properties by location and criteria
- **get_listing_details** - Get detailed information about specific properties
- **check_availability** - Check availability for specific dates
- **get_reviews** - Retrieve reviews for properties
- **get_host_info** - Get information about property hosts

### Prerequisites:

- **Node.js** installed on your system
- **Internet connection** to download the MCP server package
- **NPX** available (comes with Node.js)

### Testing the Connection:

You can test the MCP server manually by running:
```bash
npx -y @openbnb/mcp-server-airbnb
```

This will download and start the OpenBnB MCP server, which the Agent Framework then connects to for real Airbnb data.

## Initialize Azure AI Agent Client

In [ ]:
Console.WriteLine("🚀 Starting with Azure AI Foundry...\n");

// Create Azure AI Agent Client
var credential = new AzureCliCredential();

var agentChatClient = new AzureAIAgentClient(
    credential: credential,
    modelDeploymentName: modelDeploymentName,
    projectEndpoint: new Uri(projectEndpoint)
);

Console.WriteLine("✅ Azure AI Agent Client created");

## Create MCP Plugin and Agent

Now we will create the MCP plugin connection to the OpenBnB server and initialize our agent with it.

In [ ]:
Console.WriteLine("🔧 Creating MCP Plugin...\n");

// Create MCP plugin connection to real OpenBnB server
var mcpPlugin = await MCPStdioPlugin.CreateAsync(
    name: "AirbnbSearch",
    description: "Search for Airbnb accommodations using OpenBnB MCP server",
    command: "npx",
    arguments: new[] { "-y", "@openbnb/mcp-server-airbnb" }
);

Console.WriteLine("✅ MCP Plugin created and connected");

// Wait a moment for the server to fully initialize
await Task.Delay(2000);

// Try to list available tools from MCP server
try
{
    var tools = await mcpPlugin.GetToolsAsync();
    Console.WriteLine($"🔧 Available tools: {string.Join(", ", tools.Select(t => t.Name))}");
}
catch (Exception e)
{
    Console.WriteLine($"⚠️ Could not list tools: {e.Message}");
}

In [ ]:
// Create agent with MCP plugin
Console.WriteLine("\n🤖 Creating AI Agent with MCP integration...");

var agent = new ChatAgent(
    name: "AirbnbAgent",
    chatClient: agentChatClient,
    instructions: @"You are an Airbnb search assistant. Use the available functions to search for properties. 
    Format results in a clear, readable format with property name, price, rating, and link for each result.",
    plugins: new[] { mcpPlugin }
);

Console.WriteLine("✅ Agent created with MCP plugin integration");

## Running the Agent with OpenBnB MCP Server

Now we will run the AI Agent that connects to the OpenBnB MCP server to search for real Airbnb accommodations in Stockholm for 2 adults and 1 kid. Feel free to change the user input to modify the search criteria.

In [ ]:
// Execute agent with user request
var userInput = "Find Airbnb in Stockholm for 2 adults 1 kid";
Console.WriteLine($"\n🔍 User: {userInput}");
Console.WriteLine("\n🤖 Agent is processing your request...\n");

try
{
    // Invoke agent with streaming for real-time responses
    await foreach (var response in agent.InvokeStreamingAsync(userInput))
    {
        if (response.Content != null)
        {
            Console.Write(response.Content);
        }
    }
    
    Console.WriteLine("\n\n✅ Request completed successfully!");
}
catch (Exception e)
{
    Console.WriteLine($"\n❌ Error processing request: {e.Message}");
    Console.WriteLine(e.StackTrace);
}
finally
{
    // Cleanup MCP plugin
    if (mcpPlugin != null)
    {
        await mcpPlugin.DisposeAsync();
        Console.WriteLine("\n🧹 MCP Plugin cleaned up");
    }
}

## Try Another Search

Let's demonstrate the agent's capability with different search criteria.

In [ ]:
// Recreate MCP plugin for another search
var mcpPlugin2 = await MCPStdioPlugin.CreateAsync(
    name: "AirbnbSearch",
    description: "Search for Airbnb accommodations",
    command: "npx",
    arguments: new[] { "-y", "@openbnb/mcp-server-airbnb" }
);

await Task.Delay(2000);

var agent2 = new ChatAgent(
    name: "AirbnbAgent",
    chatClient: agentChatClient,
    instructions: @"You are an Airbnb search assistant. Use the available functions to search for properties. 
    Format results in a clear, readable format.",
    plugins: new[] { mcpPlugin2 }
);

var userInput2 = "Find luxury apartments in Paris for 2 guests";
Console.WriteLine($"\n🔍 User: {userInput2}");
Console.WriteLine("\n🤖 Agent is processing...\n");

try
{
    await foreach (var response in agent2.InvokeStreamingAsync(userInput2))
    {
        if (response.Content != null)
        {
            Console.Write(response.Content);
        }
    }
    
    Console.WriteLine("\n\n✅ Request completed!");
}
catch (Exception e)
{
    Console.WriteLine($"\n❌ Error: {e.Message}");
}
finally
{
    await mcpPlugin2.DisposeAsync();
    Console.WriteLine("\n🧹 Cleanup complete");
}

# Summary

Congratulations! You've successfully built an AI agent in C# using Microsoft Agent Framework that integrates with real-world accommodation search using the Model Context Protocol (MCP):

## Technologies Used:
- **Microsoft Agent Framework (C#)** - For building intelligent agents with tool-calling capabilities
- **Azure AI Foundry** - For LLM capabilities and chat completion
- **MCP (Model Context Protocol)** - For standardized tool integration
- **OpenBnB MCP Server** - For real Airbnb search functionality
- **Node.js/NPX** - For running the external MCP server

## What You've Learned:
- **MCP Integration**: Connecting Agent Framework to external MCP servers in C#
- **Real-time Data Access**: Searching actual Airbnb properties through live APIs
- **Protocol Communication**: Using stdio communication between agent and MCP server
- **Function Discovery**: Automatically discovering available functions from MCP servers
- **Streaming Responses**: Real-time streaming of agent responses
- **Resource Management**: Proper cleanup and disposal of MCP connections

## Key Differences from Semantic Kernel:
- **Simplified API**: Agent Framework provides a more streamlined agent creation process
- **Native Plugin Support**: Direct plugin integration without additional kernel configuration
- **Streaming-First**: Built-in support for streaming responses
- **Azure AI Foundry Integration**: Native support for Azure AI Foundry endpoints

## Next Steps:
- Integrate additional MCP servers (weather, flights, restaurants)
- Build a multi-agent system with MCP tool sharing
- Create custom MCP servers for your own data sources
- Implement conversation memory across multiple interactions
- Deploy the agent to Azure with orchestrated MCP servers
- Add authentication and personalized recommendations